### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ecommerce_shipping",
    dataset_year="2021",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/prachi13/customer-analytics",
    download_description="""
kaggle datasets download prachi13/customer-analytics -p local-data-warehouse/ecommerce_shipping/ && unzip local-data-warehouse/ecommerce_shipping/customer-analytics.zip -d local-data-warehouse/ecommerce_shipping/ && rm local-data-warehouse/ecommerce_shipping/customer-analytics.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{gopalani2021ecommerce,
  author       = {Prachi Gopalani},
  title        = {E-Commerce Shipping Data},
  year         = {2021},
  howpublished = {\url{https://www.kaggle.com/datasets/prachi13/customer-analytics}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="gopalani2021ecommerce",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We dropped the ID column.
- We renamed the target feature "Reached.on.Time_Y.N" to "ArrivedLate" and mapped binary values to "Yes"/"No".
- Anomaly: the target and task seems somewhat disconnected from the features. Moreover, some source information on the data is missing and there might be some translation issues.
- Anomaly: there might be some data issues related to "Warehouse_block" and the value "F" consisting of two block "E" and "F".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="ArrivedLate",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="ArrivedLate",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/Train.csv")

target_feature = "ArrivedLate"
df = df.rename(columns={"Reached.on.Time_Y.N": target_feature})
df = df.drop(columns=["ID"])
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"})

cat_features = [
    "Warehouse_block",
    "Mode_of_Shipment",
    "Product_importance",
    "Gender",
    "ArrivedLate",
]

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 10,999
Columns: 11
Use sampling: False (sample size: 10,999)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Weight_in_gms', 'Cost_of_the_Product', 'Discount_offered', 'Prior_purchases', 'Customer_care_calls', 'Warehouse_block', 'Customer_rating', 'Product_importance', 'Mode_of_Shipment', 'Gender']
Rows remaining as candidates after top-10 filter: 0 (of 10,999)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,ArrivedLate
0,F,Ship,4,5,216,3,high,M,26,2053,Yes
1,A,Road,3,1,220,3,low,F,6,5572,Yes
2,F,Flight,3,2,215,4,low,F,3,4042,No
3,D,Flight,5,1,160,5,low,F,1,4672,No
4,B,Ship,5,4,229,2,medium,F,44,2419,Yes


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Warehouse_block,category,0.0,0.0,5.0,"F, D, A, C, B"
1,Mode_of_Shipment,category,0.0,0.0,3.0,"Ship, Flight, Road"
2,Product_importance,category,0.0,0.0,3.0,"low, medium, high"
3,Gender,category,0.0,0.0,2.0,"F, M"
4,ArrivedLate,category,0.0,0.0,2.0,"Yes, No"
5,Customer_care_calls,int64,0.0,0.0,6.0,"4, 3, 5, 6, 2, 7"
6,Customer_rating,int64,0.0,0.0,5.0,"3, 1, 4, 5, 2"
7,Cost_of_the_Product,int64,0.0,0.0,215.0,"245, 257, 260, 254, 264, 243, 255, 263, 258, 266"
8,Prior_purchases,int64,0.0,0.0,8.0,"3, 2, 4, 5, 6, 10, 7, 8"
9,Discount_offered,int64,0.0,0.0,65.0,"10, 2, 6, 9, 7, 3, 4, 1, 5, 8"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Customer_care_calls,10999.0,4.054459,1.141490,2.0,7.0
Customer_rating,10999.0,2.990545,1.413603,1.0,5.0
Cost_of_the_Product,10999.0,210.196836,48.063272,96.0,310.0
Prior_purchases,10999.0,3.567597,1.522860,2.0,10.0
Discount_offered,10999.0,13.373216,16.205527,1.0,65.0
Weight_in_gms,10999.0,3634.016729,1635.377251,1001.0,7846.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column             rank                      
ArrivedLate        1        Yes   6563  59.67
                   2         No   4436  40.33
Gender             1          F   5545  50.41
                   2          M   5454  49.59
Mode_of_Shipment   1       Ship   7462  67.84
                   2     Flight   1777  16.16
                   3       Road   1760  16.00
Product_importance 1        low   5297  48.16
                   2     medium   4754  43.22
                   3       high    948   8.62
Warehouse_block    1          F   3666  33.33
                   2          D   1834  16.67
                   3          A   1833  16.67
                   4          C   1833  16.67
                   5          B   1833  16.67

In [8]:
# Target Distribution
target_df

,count,pct
ArrivedLate,,
Yes,6563,59.67
No,4436,40.33


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to ecommerce_shipping/019d5a4d-7f92-7bc7-a5fc-5a63ac679eef
019d5a4d-7f92-7bc7-a5fc-5a63ac679eef
3e69a0c5e031495fd1516f01b0fd60ffca771a1acadb0c7cce6befe39dec223e
